### 06. Парадоксальные студенты: кто выбивается из ожиданий?

Кто те, у кого высокая оценка при плохих привычках (много соцсетей, мало сна, мало учёбы)? И наоборот — кто учится много, спит достаточно, но получает низкую оценку? Что общего у этих двух групп? Есть ли у них что-то, что компенсирует или, наоборот, блокирует результат?

In [2]:
import sys
import os
from pathlib import Path

project_root = Path(os.getcwd()).parent 
sys.path.insert(0, str(project_root))

from src.preparation import load_df
df = load_df()

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import src.constants as cnst 

plt.rcParams['font.family'] = 'Arial'

df_02_06 = df.copy()

df_02_06['is_study_bad'] = df_02_06.study_hours_per_day < cnst.THRESHOLDS_BAD['study']
df_02_06['is_entertainment_bad'] = (df_02_06.social_media_hours > cnst.THRESHOLDS_BAD['social']) | (df_02_06.netflix_hours > cnst.THRESHOLDS_BAD['netflix'])
df_02_06['is_sleep_bad'] = df_02_06.sleep_hours < cnst.THRESHOLDS_BAD['sleep']

df_02_06['is_study_good'] = df_02_06.study_hours_per_day > cnst.THRESHOLDS_GOOD['study']
df_02_06['is_entertainment_good'] = (df_02_06.social_media_hours < cnst.THRESHOLDS_GOOD['social']) & (df_02_06.netflix_hours < cnst.THRESHOLDS_GOOD['netflix'])
df_02_06['is_sleep_good'] = df_02_06.sleep_hours > cnst.THRESHOLDS_GOOD['sleep']

df_02_06 = df_02_06.loc[(df_02_06.exam_score >= 80) | (df_02_06.exam_score < 50)]
df_02_06['exam_group'] = np.where(df_02_06.exam_score >= 80, 'exam_score_80_plus', 'exam_score_lower_50')

res_02_06 = df_02_06.groupby('exam_group').agg(
    study_paradox_neg=('is_study_bad', lambda i: i.sum() / i.count() * 100),
    entertainment_paradox_neg=('is_entertainment_bad', lambda i: i.sum() / i.count() * 100),
    sleep_paradox_neg=('is_sleep_bad', lambda i: i.sum() / i.count() * 100),
    study_paradox_pos=('is_study_good', lambda i: i.sum() / i.count() * 100),
    entertainment_paradox_pos=('is_entertainment_good', lambda i: i.sum() / i.count() * 100),
    sleep_paradox_pos=('is_sleep_good', lambda i: i.sum() / i.count() * 100),
).unstack().to_frame().round(1).rename(columns={0: 'pct'}).swaplevel().sort_index(level=0)

res_02_06.index = res_02_06.index.set_names(['exam_group', 'paradox'])
display(res_02_06.style.format('{:.1f}', subset=['pct']).background_gradient(subset=['pct'], cmap='Blues'))


# case study  
df_02_06.loc[(df_02_06.is_entertainment_bad) & (df_02_06.exam_group == 'exam_score_80_plus')][['student_id', 'gender', 'study_hours_per_day', 'social_media_hours', 'sleep_hours', 'attendance_percentage']]

,student_id,gender,study_hours_per_day,social_media_hours,sleep_hours,attendance_percentage
361,S1361,Male,4.4,6.1,5.1,92.6


#### 07. Выводы

В ходе анализа были выделены две крайние группы студентов: с высокой успеваемостью (80+ баллов за экзамен) и низкой (<50 баллов за экзамен), а также проверена гипотеза о существованиис студентов с парадоксальными паттернами, которые не соответствуют их оценкам.

***1. Учёба — главный и единственный предсказуемый фактор.***  
Здесь никаких парадоксов не было обнаружено: среди студентов с высокой успеваемостью 83% учатся более 4 часов в день (и 0% — менее 2 часов). Среди студентов с низкой успеваемостью картина противоположная: 62.6% имеют крайне низкую учебную нагрузку. Оценка линейно связана с тем, сколько человек учится.

***2. Сон не является определяющим фактором.***   
Среди студентов с высокой успеваемостью только 6.9% имеют сильный недосып (<5 часов), тогда как среди студентов с низкой успеваемостью таких 17.6%. Сон влияет на общий тонус, но сам по себе не гарантирует провала / успеха на экзамене.

***3. Развлечения: гипотеза о «парадоксальных» отличниках не подтвердилась.***   
Студентов, которые проводят много времени в соцсетях / за просмотром сериалов, но при этом сдают экзамены на 80+, практически нет. Доля студентов с высокой успеваемостью, имеющих «плохую» привычку по развлечениям (5+ часов), составляет всего 0.4%. Большинство отличников (16.6%) имеют хорошую цифровую гигиену — они мало времени проводят в социальных сетях и не злоупотребляют сериалами.

***4. Анализ отдельного случая (Case Study).***   
Рассмотрение конкретного профиля из группы парадоксальных студентов (студента S1361) показало, за счет чего происходит  компенсация. Студент тратит 6.1 часа в день на соцсети и спит всего 5.1 часа, однако имеет посещаемость 92.6% и учит материал 4.4 часа в день.
Это указывает на то, что высокая посещаемость и вовлеченность в учебный процесс компенсируют негативное влияние цифровых развлечений и недосыпа. «Парадоксальные» студенты с высокой успеваемостью — это не гении, которые учатся мало и сдают экзамены хорошо, а люди, которые находят в своем графике время на учебу и дисциплинированно ходят на занятия, даже если вечером проводят время в социальных сетях.